# Module 00 — Assignment: The Perceptron

> **This is a hands-on assignment, not a reading.** You implement the pieces marked
> `# TODO`, then run the check cell right below each one. When a check prints a tiny
> `rel_error` and `OK`, you got it right. The worked walkthrough lives in
> [`notebook.ipynb`](notebook.ipynb) and the canonical solution in
> [`python/perceptron.py`](python/perceptron.py) — try not to peek until you've attempted it.

**What you'll build:** the simplest thing that can *learn* — a neuron that computes a
weighted sum and thresholds it, trained by Rosenblatt's 1957 rule. You'll watch it
conquer two separable blobs, then hit the **XOR wall** that forces us into hidden layers
(Module 01).

**How to work through it**
1. Run the setup cell once.
2. For each `# TODO … END OF YOUR CODE` block: write your code *between* the delimiter
   lines, then run the check cell that follows.
3. Answer each **Inline Question** in the blank provided.
4. Finish by running the final self-check — it asserts your results match the answer key
   to `1e-9`.

**On AI use:** treat any AI assistant like a tutor, not an autocomplete — use it to
understand *why* the update rule works, not to paste in the three lines you're meant to derive.

In [ ]:
# ============================================================================
# Setup — run this once. Nothing to implement here.
# ============================================================================
import numpy as np
import os, sys

# Make the check harness and the canonical answer key importable, whether you
# launched Jupyter from the repo root or from this module's folder.
def _find(subdir, filename):
    for cand in (subdir,
                 os.path.join("topics", "00-perceptron", subdir),
                 os.path.join(os.getcwd(), subdir)):
        if os.path.exists(os.path.join(cand, filename)):
            return cand
    raise FileNotFoundError(
        f"could not locate {subdir}/{filename}; launch Jupyter from the repo root "
        f"or from topics/00-perceptron/")

sys.path.insert(0, _find("tests", "check_utils.py"))
sys.path.insert(0, _find("python", "perceptron.py"))
from check_utils import rel_error, compare_to_canonical
import perceptron  # the canonical answer key (python/perceptron.py)

# ---------------------------------------------------------------------------
# GIVEN: a deterministic 64-bit RNG and the toy dataset. These are identical to
# the canonical C/Python source, so your results will match the answer key
# bit-for-bit. Do NOT reimplement them — that would break the comparison.
# ---------------------------------------------------------------------------
_MASK64 = (1 << 64) - 1

class Rng:
    """64-bit linear congruential generator (matches the C code exactly)."""
    def __init__(self, seed):
        self.state = seed & _MASK64
    def next_u64(self):
        self.state = (self.state * 6364136223846793005 + 1442695040888963407) & _MASK64
        return self.state
    def uniform(self):            # [0, 1)
        return (self.next_u64() >> 11) * (1.0 / 9007199254740992.0)
    def signed(self):            # [-1, 1)
        return 2.0 * self.uniform() - 1.0

N_PER_CLASS = 50
N_SAMPLES   = 2 * N_PER_CLASS
SPREAD      = 1.5

def make_blobs(rng):
    """Two 2D blobs: class 0 near (-2,-2), class 1 near (+2,+2). Separable."""
    centers = ((-2.0, -2.0), (2.0, 2.0))
    X = np.zeros((N_SAMPLES, 2), dtype=np.float64)
    y = np.zeros(N_SAMPLES, dtype=np.int64)
    i = 0
    for c in (0, 1):
        for _ in range(N_PER_CLASS):
            X[i, 0] = centers[c][0] + SPREAD * rng.signed()
            X[i, 1] = centers[c][1] + SPREAD * rng.signed()
            y[i] = c
            i += 1
    return X, y

print("setup ok — numpy", np.__version__)

## Part 1 — The neuron: pre-activation and prediction

A perceptron holds two weights $w_0, w_1$ and a bias $b$. Given an input $(x_0, x_1)$ it
forms the **pre-activation**

$$ z = w_0 x_0 + w_1 x_1 + b $$

and then **fires** — predicts class 1 — when $z \ge 0$, otherwise predicts 0. That threshold
is the *step activation*:

$$ \hat y = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases} $$

Geometrically, $z = 0$ is a straight line; the neuron labels everything on one side 1 and
the other side 0.

**Implement** `preactivation` and `predict` in the `Perceptron` class below.

In [ ]:
class Perceptron:
    """A single neuron: two weights and a bias, all starting at zero."""
    def __init__(self):
        self.w0 = 0.0
        self.w1 = 0.0
        self.b  = 0.0

    def preactivation(self, x0, x1):
        z = None
        #######################################################################
        # TODO: Compute the pre-activation z = w0*x0 + w1*x1 + b and store it  #
        # in `z`. One line — the weighted sum plus the bias.                  #
        #######################################################################

        #######################################################################
        #                          END OF YOUR CODE                           #
        #######################################################################
        return z

    def predict(self, x0, x1):
        yhat = None
        #######################################################################
        # TODO: Return 1 when the pre-activation is >= 0, else 0. Reuse the    #
        # preactivation() method you just wrote. (This is the step function.)  #
        #######################################################################

        #######################################################################
        #                          END OF YOUR CODE                           #
        #######################################################################
        return yhat

In [ ]:
# Check: set known weights w=(1,1), b=0, so the boundary is the line x0 + x1 = 0.
# Points above the line fire (1); points below don't (0); points on it fire (>= 0).
p = Perceptron()
p.w0, p.w1, p.b = 1.0, 1.0, 0.0

pts      = [(2.0, 2.0), (-2.0, -2.0), (1.0, -0.5), (-1.0, 0.5), (0.0, 0.0)]
expect_z = [ 4.0,        -4.0,         0.5,          -0.5,        0.0      ]
expect_y = [ 1,          0,            1,            0,           1        ]

got_z = [p.preactivation(x0, x1) for x0, x1 in pts]
got_y = [p.predict(x0, x1)       for x0, x1 in pts]

print("pre-activation rel_error:", rel_error(got_z, expect_z))   # want ~0
print("predicted labels:", got_y, "  expected:", expect_y)
assert rel_error(got_z, expect_z) < 1e-12, "preactivation is off — check w0*x0 + w1*x1 + b"
assert got_y == expect_y, "predict is off — the boundary is z >= 0, not z > 0"
print("Part 1 OK ✓")

### Inline Question 1

The boundary is the line $w_0 x_0 + w_1 x_1 + b = 0$.

1. If you increase the bias $b$ (keeping the weights fixed), which way does the boundary
   move — and why?
2. We fire when $z \ge 0$. Would anything meaningful change if we instead fired on $z > 0$?
   Consider a point that lands *exactly* on the line.

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Part 2 — Rosenblatt's learning rule

We start with all weights at zero (a boundary that classifies nothing) and learn from
mistakes. For each sample, predict, then nudge the weights **toward the correct answer**
in proportion to the error:

$$ \text{err} = y - \hat y \in \{-1, 0, +1\} $$
$$ w_0 \mathrel{+}= \eta\,\text{err}\,x_0, \quad
   w_1 \mathrel{+}= \eta\,\text{err}\,x_1, \quad
   b   \mathrel{+}= \eta\,\text{err} $$

where $\eta$ is the learning rate. When the prediction is already right, $\text{err}=0$ and
nothing moves — the update is **mistake-driven**. On linearly separable data this is
guaranteed to converge (Rosenblatt, 1962).

`accuracy` is given (it only needs your `predict`). **Implement the single update inside**
`train_epoch`.

In [ ]:
def accuracy(p, X, y, n):
    """GIVEN: fraction of the first n samples that p classifies correctly."""
    correct = 0
    for i in range(n):
        if p.predict(float(X[i, 0]), float(X[i, 1])) == int(y[i]):
            correct += 1
    return correct / n

def train_epoch(p, lr, X, y, n):
    """One pass of Rosenblatt's rule over all n samples, in fixed order."""
    for i in range(n):
        x0 = float(X[i, 0])
        x1 = float(X[i, 1])
        #######################################################################
        # TODO: Perform one perceptron update for sample i:                   #
        #   1. yhat = p.predict(x0, x1)                                        #
        #   2. err  = int(y[i]) - yhat          # one of -1, 0, +1            #
        #   3. p.w0 += lr * err * x0                                           #
        #      p.w1 += lr * err * x1                                           #
        #      p.b  += lr * err                                               #
        # If err == 0 the sample is already correct and nothing changes.      #
        #######################################################################
        pass
        #######################################################################
        #                          END OF YOUR CODE                           #
        #######################################################################

In [ ]:
# Train on the separable blobs for 20 epochs; it should reach 100% accuracy.
lr, epochs = 0.1, 20
rng = Rng(42)
X, y = make_blobs(rng)
p = Perceptron()
for e in range(epochs):
    train_epoch(p, lr, X, y, N_SAMPLES)
acc = accuracy(p, X, y, N_SAMPLES)
print(f"after {epochs} epochs: acc={acc:.3f}  w=({p.w0:.4f}, {p.w1:.4f})  b={p.b:.4f}")
assert acc == 1.0, "expected 100% on separable blobs — check your train_epoch update"

# Now compare every number to the canonical answer key (python/perceptron.py).
print("\nchecking against the answer key:")
compare_to_canonical((p.w0, p.w1, p.b, acc), perceptron.run_blobs(),
                     labels=["w0", "w1", "b", "acc"])

In [ ]:
# GIVEN: see the boundary your perceptron learned.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#d55e00", edgecolor="k", s=30, label="class 0")
ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#0072b2", edgecolor="k", s=30, label="class 1")

# boundary: w0*x0 + w1*x1 + b = 0  ->  x1 = -(w0*x0 + b) / w1
xs = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
if abs(p.w1) > 1e-12:
    ax.plot(xs, -(p.w0 * xs + p.b) / p.w1, "k--", label="learned boundary")

ax.set_title("Perceptron decision boundary")
ax.set_xlabel("x0"); ax.set_ylabel("x1")
ax.set_aspect("equal"); ax.legend()
plt.show()

### Inline Question 2

The blobs are centred at $(-2,-2)$ and $(+2,+2)$, so the "ideal" separating line is
$x_0 + x_1 = 0$ (i.e. weights proportional to $(1, 1)$ with bias near 0).

1. Look at the printed weights — are they close to a scalar multiple of $(1, 1)$?
2. The perceptron rule stops updating the moment every point is classified correctly.
   Does that mean it finds the *best* (widest-margin) line, or just *a* line that works?

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Part 3 — The wall: XOR

XOR labels a point 1 when *exactly one* of its coordinates is 1:

| $x_0$ | $x_1$ | $y$ |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No single straight line can put the two 1s on one side and the two 0s on the other. Run
the same training you just wrote on XOR and watch the accuracy **refuse to reach 1.0**.

In [ ]:
# GIVEN: same rule, same code — but XOR is not linearly separable.
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
y_xor = np.array([0, 1, 1, 0], dtype=np.int64)

p_xor = Perceptron()
for _ in range(100):
    train_epoch(p_xor, 0.1, X_xor, y_xor, 4)
acc_xor = accuracy(p_xor, X_xor, y_xor, 4)

print(f"XOR accuracy after 100 epochs: {acc_xor:.3f}   "
      f"w=({p_xor.w0:.3f}, {p_xor.w1:.3f})  b={p_xor.b:.3f}")
print("Stuck below 1.0 — one straight line cannot separate XOR. This is 'the wall'.")
assert acc_xor < 1.0, "if XOR hit 100%, something is wrong — it should be impossible here"

### Inline Question 3

1. In one sentence, *why* can no single perceptron solve XOR? (Think about what shape a
   perceptron's decision region always is.)
2. What is the smallest change to the model that would let it separate XOR? This is exactly
   the door into [Module 01](../01-mlp-backprop/).

$\color{blue}{\textit{Your answer:}}$ *fill this in*

## Verify against the answer key

The point of this project: your from-scratch results must match the canonical source of
truth (`python/perceptron.py`), which the C↔Python agreement test also pins to full double
precision. If anything below fails, revisit the `# TODO` it points back to.

In [ ]:
print("blobs:")
compare_to_canonical((p.w0, p.w1, p.b, acc), perceptron.run_blobs(),
                     labels=["w0", "w1", "b", "acc"])
print("\nxor:")
compare_to_canonical((p_xor.w0, p_xor.w1, p_xor.b, acc_xor), perceptron.run_xor(),
                     labels=["w0", "w1", "b", "acc"])
print("\nAll checks passed — your perceptron matches the answer key. 🎉")